# Flashcard Generator

Give it a topic (e.g. "the French Revolution") or paste a block of text, and get back 5
question/answer flashcards.

Claude is instructed to respond using XML tags, which we parse and print cleanly, numbered
(the raw API response is never shown to the user).

In [1]:
%pip install -q anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import os
from textwrap import dedent
from dotenv import load_dotenv
from anthropic import Anthropic

In [3]:
load_dotenv()

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
model = "claude-sonnet-4-6"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 3000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
SYSTEM_PROMPT = dedent("""
    You are a flashcard generator. Given a topic or a block of text, create exactly 5
    flashcards that test understanding of the most important concepts. Respond with
    structured output and nothing else — no preamble, no closing remarks, no text outside
    the tags below.

    Respond using exactly this format:

    <flashcards>
    <card>
    <question>First question.</question>
    <answer>First answer.</answer>
    </card>
    <card>
    <question>Second question.</question>
    <answer>Second answer.</answer>
    </card>
    (continue for a total of 5 cards)
    </flashcards>

    Rules:
    - Generate exactly 5 <card> elements — no more, no fewer.
    - Each <card> must contain exactly one <question> and one <answer>.
    - Questions should test genuine understanding, not trivial word-matching.
    - Answers should be concise but complete — one to three sentences.
    - Do not wrap the output in markdown code fences.
    - Do not include any text before <flashcards> or after </flashcards>.
""").strip()

In [5]:
def get_flashcards(topic_or_text):
    messages = []
    add_user_message(messages, topic_or_text)
    return chat(messages, system=SYSTEM_PROMPT, temperature=0)

In [6]:
def parse_flashcards(raw):
    card_blocks = re.findall(r"<card>(.*?)</card>", raw, re.DOTALL)

    flashcards = []
    for block in card_blocks:
        question_match = re.search(r"<question>(.*?)</question>", block, re.DOTALL)
        answer_match = re.search(r"<answer>(.*?)</answer>", block, re.DOTALL)

        flashcards.append({
            "question": question_match.group(1).strip() if question_match else None,
            "answer": answer_match.group(1).strip() if answer_match else None,
        })

    return flashcards

In [7]:
def print_flashcards(flashcards):
    if not flashcards:
        print("(no flashcards returned)")
        return

    for i, card in enumerate(flashcards, 1):
        print(f"Card {i}")
        print("-" * 60)
        print(f"Q: {card['question'] or '(no question returned)'}")
        print(f"A: {card['answer'] or '(no answer returned)'}")
        print()

In [8]:
topic_or_text = input("Enter a topic or paste text to generate flashcards from: ")

raw_response = get_flashcards(topic_or_text)
flashcards = parse_flashcards(raw_response)
print_flashcards(flashcards)

Card 1
------------------------------------------------------------
Q: What is the fundamental difference between supervised and unsupervised learning in terms of training data?
A: Supervised learning uses labeled training data, where each input has a corresponding correct output, while unsupervised learning uses unlabeled data and must find patterns or structure on its own without any predefined correct answers.

Card 2
------------------------------------------------------------
Q: Why is supervised learning called "supervised," and what role does the "supervisor" play?
A: It is called supervised because a teacher or supervisor provides the correct answers (labels) during training. The algorithm learns by comparing its predictions to these correct answers and adjusting to minimize errors.

Card 3
------------------------------------------------------------
Q: What types of problems are typically solved by supervised learning versus unsupervised learning?
A: Supervised learning is use